# Metode Baru: Validasi Viral, Estimasi Waktu Viral, dan Ekstraksi Multi-Tempat

Notebook ini adalah **lanjutan/pelengkap** dari `scrap_tempat_viral.ipynb` dan `scrap_tempat_viral_extend.ipynb`. Notebook-notebook itu tidak diubah — notebook ini memakai data yang sudah dikumpulkan (`df_news.csv`) dan mengganti/menambah langkah **ekstraksi & penilaian viral** untuk menjawab 3 kesulitan yang dialami:

| # | Kesulitan | Solusi di notebook ini |
| - | --- | --- |
| 1 | Validitas "viral" rendah — artikel sering hanya berisi tempat mainstream/rekomendasi, bukan yang benar-benar ramai dibahas | Skor `viral_confidence_score` gabungan dari **(a)** bahasa eksplisit "viral/ramai dibahas" di artikel (dinilai LLM, bukan regex), **(b)** jumlah *sumber berita unik* yang memberitakan tempat yang sama dalam rentang waktu pendek (proxy "ramai dibahas" tanpa perlu API media sosial berbayar), **(c)** lonjakan (spike) minat pencarian di **Google Trends** (gratis, via `pytrends`) untuk nama tempat tersebut |
| 2 | Tanggal mulai viral sulit dicari | `viral_since_estimate` dipilih berdasarkan prioritas: tanggal eksplisit yang disebut artikel → tanggal *spike* Google Trends → tanggal artikel paling awal dalam klaster pemberitaan. Kolom `viral_since_source` mencatat metode mana yang dipakai supaya transparan |
| 3 | Ekstraksi tempat dari artikel sering tidak lengkap (regex berbasis judul saja) | Ekstraksi diganti dengan **LLM (Claude) structured extraction** yang membaca seluruh isi artikel dan mengembalikan **daftar semua tempat** yang disebut (bukan cuma 1 dari judul), lengkap dengan lokasi, alasan viral, dan indikasi eksplisit/tidaknya viral. Ada fallback regex multi-tempat bila tidak ada API key |

Alur kerja:

1. Muat `df_news.csv` (tidak fetch ulang).
2. Ekstraksi multi-tempat per artikel (LLM, fallback regex).
3. Normalisasi & deduplikasi nama tempat antar-artikel (fuzzy matching) → satu baris per tempat unik.
4. Hitung sinyal viral: diversitas sumber, klaster waktu pemberitaan, Google Trends.
5. Hitung `viral_confidence_score` & `viral_since_estimate`, tandai kandidat yang "cuma mainstream" untuk direview manual.
6. Geocoding tempat yang lolos threshold (pakai fungsi yang sama seperti notebook utama).
7. Simpan hasil akhir + catatan keterbatasan.

## 1. Install & Import Library

Tambahan dari notebook utama: `anthropic` (LLM extraction), `rapidfuzz` (fuzzy dedup nama tempat).

In [1]:
%pip install pandas requests geopy pytrends rapidfuzz anthropic python-dateutil tqdm -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import glob
import json
import os
import re
import time
from datetime import datetime, timedelta

import pandas as pd
from dateutil import parser as dateparser
from rapidfuzz import fuzz
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)

c:\Users\PC NEVIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Muat Data Berita

Tidak fetch ulang — memakai `df_news.csv` (hasil `scrap_tempat_viral.ipynb`), buang duplikat berdasarkan `final_url`, dan buang artikel yang isinya terlalu pendek.

In [3]:
NEWS_CSV_FILE = "df_news.csv"
MIN_ARTICLE_LEN = 150

df_news_all = pd.read_csv(NEWS_CSV_FILE)
print("File dipakai:", NEWS_CSV_FILE)

# beberapa batch mungkin punya nama kolom isi artikel yang sedikit berbeda; standardisasi
text_col_candidates = [c for c in ["article_text", "article_text_extended", "text"] if c in df_news_all.columns]
if "article_text" not in df_news_all.columns and text_col_candidates:
    df_news_all["article_text"] = df_news_all[text_col_candidates[0]]

dedup_key = "final_url" if "final_url" in df_news_all.columns else "link"
df_news_all = df_news_all.dropna(subset=[dedup_key])
df_news_all = df_news_all.drop_duplicates(subset=[dedup_key]).reset_index(drop=True)
df_news_all = df_news_all[df_news_all["article_text"].fillna("").str.len() >= MIN_ARTICLE_LEN].reset_index(drop=True)

print("Total artikel unik & layak diproses:", len(df_news_all))
df_news_all

File dipakai: df_news.csv
Total artikel unik & layak diproses: 124


,keyword,title,link,published,source,final_url,article_text,article_text_length
0,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat Makan - Jatimnet.com,https://news.google.com/rss/articles/CBMifkFVX3lxTE9qZDRrbTc5elBjLU5ra0pZVnp1M21wMFRZd1NydUpGY2xlbm5vMktZZEJXNnhfcVZ...,2026-09-25 08:00:00+00:00,Jatimnet.com,https://jatimnet.com/media-sosial-mengubah-cara-kita-memilih-tempat-makan,"Jumat, 25 September 2026 08:00 UTC Ilustrasi: Pilihan dari Layar. -Dx Gen-AI JATIMNET.COM - Media sosial mengubah ca...",7110
1,tempat makan viral,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legendaris di Jakarta dan Bogor - food.detik.com,https://news.google.com/rss/articles/CBMitgFBVV95cUxOYk92ckFweFFab1BZMHdPUTB2U2NXeE5vOVpHOUJfbXVqUm5YeHNHVjVUdUYxT2x...,2026-09-26 08:00:30+00:00,food.detik.com,https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor,"Di Jakarta dan Bogor, ada kedai soto mie yang sudah buka sejak puluhan tahun lalu dan terkenal lezat. Bahkan sampai ...",2855
2,tempat makan viral,Rekomendasi 7 Tempat Makan Ramah Kantong di Jogja dan Tips Menggunakan GrabFood - RRI.co.id,https://news.google.com/rss/articles/CBMiwgFBVV95cUxPQzJKTVJPcXJKa3ZnWnhCTXZUYVBySTd5d25iek4xRmdETXZXdWFicVJXdkl4ZGF...,2026-09-23 06:38:00+00:00,RRI.co.id,https://rri.co.id/yogyakarta/kuliner/2752639/rekomendasi-7-tempat-makan-ramah-kantong-di-jogja-dan-tips-menggunakan-...,"RRI.CO.ID, Yogyakarta - Daerah Istimewa Yogyakarta (DIY).dikenal luas sebagai surganya kuliner murah di Indonesia, t...",6348
3,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ada 40 Lebih Tenant hingga Kuliner Viral - aktualita.co",https://news.google.com/rss/articles/CBMidEFVX3lxTE5xMXhUUkZOTGJJcWZ2VmgwdGgtcnJsZ3NnbklPWENaLWV0RTVpM2Njc25KNDhpOFZ...,2026-09-27 10:22:00+00:00,aktualita.co,https://aktualita.co/ragam/tiktok-food-fest-2026-hari-terakhir-gbk,"English edition available TikTok Food Fest 2026 Hari Terakhir di GBK, Ada 40 Lebih Tenant hingga Kuliner Viral Jakar...",6105
4,tempat makan viral,10 Tempat Makan Viral di Bandung 2026 - nibble.id,https://news.google.com/rss/articles/CBMiaEFVX3lxTE9pZzU0UEt2M3VSYUxyTVBQdWtIMDEzaDhmRld1eWFGWk5ZaXc4Q0F0Q1FlQktpQW9...,2026-06-09 07:00:00+00:00,nibble.id,https://www.nibble.id/tempat-makan-viral-di-bandung-2026/,"Harus siapkan mental buat antre, ini dia tempat makan viral di Bandung 2026 yang sampai kini selalu gak ada habisnya...",9858
...,...,...,...,...,...,...,...,...
119,tempat wisata baru viral,"Viral, Canyoning Banyak Digemari Kalangan Muda dan Jadi Destinasi Baru di Bogor - Chanelmuslim.com",https://news.google.com/rss/articles/CBMirwFBVV95cUxOb01xLWZHc3NTXzVPSTM5SkRDSnp4VF9EMkpENlFaRXlhSjF3NGNNWllrVzVsQUN...,2026-08-24 07:00:00+00:00,Chanelmuslim.com,https://chanelmuslim.com/wisata/viral-canyoning-banyak-digemari-kalangan-muda-dan-jadi-destinasi-baru-di-bogor,"<strong>BOGOR</strong> kembali menghadirkan destinasi wisata baru yang tengah viral di kalangan muda, yakni Canyonin...",2922
120,tempat wisata baru viral,10 Tempat Wisata Hits di Sukabumi 2026 yang Wajib Masuk Bucket List Kamu! - Radar Sukabumi,https://news.google.com/rss/articles/CBMiqgFBVV95cUxQSk9ZajJRNFdtX0szU0FaLXJsMEYzVmp6V1B2d2JoV1JFcFhLeUUzUm94SGlyUlk...,2026-06-20 07:00:00+00:00,Radar Sukabumi,https://radarsukabumi.com/wisata/10-tempat-wisata-hits-di-sukabumi-2026-yang-wajib-masuk-bucket-list-kamu/,"Biar kalian nggak kudet, yuk simak 10 tempat wisata favorit di Sukabumi terbaru tahun 2026 yang wajib kalian datangi...",3528
121,tempat wisata baru viral,5 Fakta Uji Coba Berbahaya Mikutopia: Pengunjung Tewas-Wahana Patah - detikcom,https://news.google.com/rss/articles/CBMisAFBVV95cUxNTzFlNlF3Nkg4Yk9DdndobU1BQVFyR3Q5c3Y2QU42UmJZd3dJZndiT255ekF5UHh...,2026-04-05 07:00:00+00:00,detikcom,https://www.detik.com/jatim/wisata/d-8429886/5-fakta-uji-coba-berbahaya-mikutopia-pengunjung-tewas-wahana-patah,"Destinasi wisata baru Mikutopia di Kecamatan Bumiaji, Kota Batu, mendadak viral. Namun bukan ka

## 3. Ekstraksi Multi-Tempat per Artikel

**Kesulitan #3 (ekstraksi tidak lengkap)** diatasi dengan meminta LLM membaca seluruh isi artikel dan mengembalikan **semua** tempat yang disebut (bisa 1 artikel = banyak tempat, misal artikel "7 Kafe Viral di Bandung"), bukan cuma 1 tempat dari judul.

Skema output per tempat:

- `nama_tempat`
- `kategori` (Kafe/Resto/Wisata Alam/Museum/Wahana/Pameran/dll)
- `lokasi_text` — lokasi mentah seperti disebut artikel (kelurahan/kecamatan/kota/provinsi apa adanya)
- `alasan_viral` — ringkasan alasan/daya tarik
- `indikasi_waktu` — tanggal/bulan/periode eksplisit yang disebut terkait mulai viral (atau `null` jika tidak ada)
- `is_explicitly_viral` — **true** hanya jika artikel benar-benar menyatakan tempat ini viral/ramai dibahas/jadi perbincangan (bukan sekadar "rekomendasi"/"wajib coba")
- `confidence` — 0-1, seberapa yakin LLM dengan hasil ekstraksi ini

Butuh `ANTHROPIC_API_KEY` di environment variable. **Tanpa API key, notebook tetap berjalan** memakai fallback regex multi-tempat (kurang akurat, tapi tidak butuh biaya) — lihat Bagian 3b.

In [ ]:
LLM_MODEL = "claude-sonnet-5"
USE_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Menggunakan LLM extraction:", USE_LLM)

EXTRACT_TOOL_SCHEMA = {
    "name": "extract_places",
    "description": "Ekstrak semua tempat makan/wisata/hiburan yang disebut dalam artikel berita, beserta indikasi viralnya.",
    "input_schema": {
        "type": "object",
        "properties": {
            "places": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "nama_tempat": {"type": "string"},
                        "kategori": {"type": "string", "description": "Kafe/Restoran/Wisata Alam/Museum/Wahana/Pameran/lainnya"},
                        "lokasi_text": {"type": "string", "description": "Lokasi mentah apa adanya sesuai artikel (kelurahan/kecamatan/kota/kabupaten/provinsi jika disebut)"},
                        "alasan_viral": {"type": "string"},
                        "indikasi_waktu": {"type": ["string", "null"], "description": "Tanggal/bulan/periode eksplisit terkait mulai viral, atau null jika tidak disebut"},
                        "is_explicitly_viral": {"type": "boolean", "description": "true hanya jika artikel eksplisit menyebut tempat INI sedang viral/ramai dibahas/jadi perbincangan warganet, bukan sekadar direkomendasikan"},
                        "confidence": {"type": "number"},
                    },
                    "required": ["nama_tempat", "kategori", "lokasi_text", "alasan_viral", "is_explicitly_viral", "confidence"],
                },
            }
        },
        "required": ["places"],
    },
}

if USE_LLM:
    import anthropic
    client = anthropic.Anthropic()


def extract_places_llm(title: str, article_text: str, max_chars: int = 6000):
    """Ekstrak daftar tempat dari 1 artikel memakai Claude, hasil terstruktur lewat tool use."""
    text = article_text[:max_chars]
    prompt = (
        f"Judul artikel: {title}\n\nIsi artikel:\n{text}\n\n"
        "Ekstrak semua tempat makan/kafe/resto/wisata/museum/wahana/pameran yang disebutkan "
        "memakai tool extract_places. Jika artikel tidak menyebut tempat spesifik, kembalikan places kosong."
    )
    resp = client.messages.create(
        model=LLM_MODEL,
        max_tokens=2000,
        tools=[EXTRACT_TOOL_SCHEMA],
        tool_choice={"type": "tool", "name": "extract_places"},
        messages=[{"role": "user", "content": prompt}],
    )
    for block in resp.content:
        if block.type == "tool_use" and block.name == "extract_places":
            return block.input.get("places", [])
    return []

### 3b. Fallback Tanpa LLM (Regex Multi-Tempat)

Kalau tidak ada `ANTHROPIC_API_KEY`, tetap coba ambil **lebih dari 1 tempat per artikel** (bukan cuma dari judul seperti notebook lama) dengan mencari pola daftar bernomor/list (`1. Nama Tempat`, `- Nama Tempat`) yang biasa dipakai artikel "Rekomendasi N tempat viral", ditambah judul sebagai kandidat tambahan. Ini tetap heuristik dan kurang akurat dibanding LLM — dianggap sebagai *best-effort fallback*, bukan solusi utama.

In [ ]:
LIST_ITEM_PATTERN = re.compile(r"(?:^|\n)\s*(?:\d{1,2}[\.\)]|[-•])\s*([A-Z][\w'&\.\-]*(?:\s+[A-Z\w'&\.\-]+){0,5})", re.MULTILINE)
VIRAL_WORDS = re.compile(r"viral|ramai\s+dibahas|jadi\s+perbincangan|heboh|trending", re.IGNORECASE)
MONTH_NAMES = "januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember"
DATE_PATTERN = re.compile(rf"(sejak\s+)?(\d{{1,2}}\s+)?({MONTH_NAMES})\s+\d{{4}}", re.IGNORECASE)


def extract_places_fallback(title: str, article_text: str):
    candidates = {}
    for m in LIST_ITEM_PATTERN.finditer(article_text):
        name = m.group(1).strip(" .")
        if 2 <= len(name.split()) <= 6 and name not in candidates:
            candidates[name] = None
    if not candidates:
        # fallback ke judul, konsisten dengan notebook utama
        title_name = re.split(r"\s+-\s+|\s+di\s+", title)[0].strip()
        candidates[title_name] = None

    date_match = DATE_PATTERN.search(article_text)
    indikasi_waktu = date_match.group(0) if date_match else None
    is_viral = bool(VIRAL_WORDS.search(article_text))

    places = []
    for name in candidates:
        places.append({
            "nama_tempat": name,
            "kategori": "tidak diketahui",
            "lokasi_text": "",
            "alasan_viral": "",
            "indikasi_waktu": indikasi_waktu,
            "is_explicitly_viral": is_viral,
            "confidence": 0.3,
        })
    return places

### 3c. Alternatif: Ekstraksi Manual via Chat Claude.ai (Tanpa API Key)

Kalau kamu sudah punya **Claude Pro** tapi belum punya **Anthropic API key** (dua hal ini terpisah — Pro tidak otomatis memberi akses API), kamu tetap bisa memakai kualitas ekstraksi LLM dengan cara manual: generate prompt per-batch artikel di sini, tempel ke chat Claude.ai, lalu tempel balik hasil JSON-nya.

Langkahnya:

1. Jalankan sel di bawah — ini membuat file `manual_llm_batches/batch_001_prompt.txt`, `batch_002_prompt.txt`, dst (tiap file berisi beberapa artikel + instruksi format JSON).
2. Buka **chat baru** di claude.ai, salin **seluruh isi** satu file `batch_XXX_prompt.txt`, tempel sebagai pesan, kirim.
3. Salin **balasan JSON** dari Claude (pastikan hanya JSON, tanpa teks tambahan), simpan sebagai `manual_llm_batches/batch_XXX_response.json` (nama harus cocok dengan nomor batch-nya).
4. Ulangi untuk semua batch, lalu jalankan sel "Muat Hasil Manual ke Checkpoint" di bawah.
5. Setelah itu, **lanjut ke Bagian 4** langsung ke sel pembacaan checkpoint (skip sel loop otomatis) untuk membangun `df_places_raw`.

Jika artikelnya banyak, proses ini memang manual dan makan waktu — cocok untuk memvalidasi kualitas LLM dulu di sebagian data sebelum memutuskan berlangganan API.

In [ ]:
MANUAL_BATCH_DIR = "manual_llm_batches"
MANUAL_BATCH_SIZE = 8       # jumlah artikel per file prompt - kecilkan jika artikel panjang
MANUAL_MAX_CHARS = 3000     # potong isi artikel supaya prompt tidak terlalu panjang untuk chat

os.makedirs(MANUAL_BATCH_DIR, exist_ok=True)

# simpan pemetaan article_id -> metadata artikel, dipakai lagi saat mengimpor hasil balasan
id_map = {}
for idx, row in df_news_all.iterrows():
    id_map[str(idx)] = {
        "final_url": row[dedup_key],
        "title": row.get("title", ""),
        "source": row.get("source", ""),
        "published": row.get("published", ""),
    }
with open(os.path.join(MANUAL_BATCH_DIR, "id_map.json"), "w", encoding="utf-8") as f:
    json.dump(id_map, f, ensure_ascii=False, indent=2)

PROMPT_HEADER = """Kamu akan menerima beberapa artikel berita. Untuk SETIAP artikel, ekstrak semua tempat makan/kafe/resto/wisata/museum/wahana/pameran yang disebutkan.

Balas HANYA dengan JSON valid (tanpa penjelasan, tanpa markdown code fence) dengan format:
{
  "results": [
    {
      "article_id": <id artikel, angka>,
      "places": [
        {
          "nama_tempat": "...",
          "kategori": "Kafe/Restoran/Wisata Alam/Museum/Wahana/Pameran/lainnya",
          "lokasi_text": "lokasi mentah apa adanya sesuai artikel (kelurahan/kecamatan/kota/kabupaten/provinsi jika disebut)",
          "alasan_viral": "ringkasan alasan/daya tarik",
          "indikasi_waktu": "tanggal/bulan/periode eksplisit terkait mulai viral, atau null jika tidak disebut",
          "is_explicitly_viral": true atau false (true HANYA jika artikel eksplisit menyebut tempat ini viral/ramai dibahas/jadi perbincangan warganet, bukan sekadar direkomendasikan),
          "confidence": angka 0-1
        }
      ]
    }
  ]
}

Jika sebuah artikel tidak menyebut tempat spesifik, kembalikan places: [] untuk artikel itu. Sertakan SEMUA article_id yang diberikan di bawah, jangan ada yang terlewat.
"""

article_ids = list(df_news_all.index)
batches = [article_ids[i:i + MANUAL_BATCH_SIZE] for i in range(0, len(article_ids), MANUAL_BATCH_SIZE)]

for b_num, batch_ids in enumerate(batches, start=1):
    parts = [PROMPT_HEADER, "\nArtikel:\n"]
    for aid in batch_ids:
        row = df_news_all.loc[aid]
        text = str(row.get("article_text", ""))[:MANUAL_MAX_CHARS]
        parts.append(f"\n### Artikel {aid}\nJudul: {row.get('title', '')}\nIsi:\n{text}\n")
    prompt_text = "\n".join(parts)
    fname = os.path.join(MANUAL_BATCH_DIR, f"batch_{b_num:03d}_prompt.txt")
    with open(fname, "w", encoding="utf-8") as f:
        f.write(prompt_text)

print(f"Dibuat {len(batches)} file prompt di folder '{MANUAL_BATCH_DIR}/'")
print("Total artikel:", len(article_ids))
print("Contoh file:", os.path.join(MANUAL_BATCH_DIR, "batch_001_prompt.txt"))

**Setelah semua file `batch_XXX_response.json` tersimpan** di folder `manual_llm_batches/` (isi setiap file harus JSON valid seperti contoh format di atas), jalankan sel berikut untuk memuat semuanya ke `places_raw_checkpoint.jsonl` — file checkpoint yang sama dipakai Bagian 4, jadi kamu bisa langsung lanjut ke sel pembacaan checkpoint di Bagian 4 tanpa menjalankan sel loop otomatis (Opsi A/B) sama sekali.

In [ ]:
CHECKPOINT_FILE = "places_raw_checkpoint.jsonl"  # sama dengan konstanta di Bagian 4

response_files = sorted(glob.glob(os.path.join(MANUAL_BATCH_DIR, "batch_*_response.json")))
print("File respons ditemukan:", len(response_files))

with open(os.path.join(MANUAL_BATCH_DIR, "id_map.json"), encoding="utf-8") as f:
    id_map = json.load(f)

already_done_manual = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        for line in f:
            try:
                already_done_manual.add(json.loads(line)["_article_key"])
            except Exception:
                continue

written = 0
with open(CHECKPOINT_FILE, "a", encoding="utf-8") as out_f:
    for rf in response_files:
        with open(rf, encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError as e:
                print("Gagal parse", rf, "-> pastikan isinya JSON valid tanpa teks tambahan:", e)
                continue
        for result in data.get("results", []):
            aid = str(result.get("article_id"))
            meta = id_map.get(aid)
            if not meta:
                print("article_id tidak dikenal:", aid, "di", rf)
                continue
            key = meta["final_url"]
            if key in already_done_manual:
                continue
            places = result.get("places", [])
            for p in places:
                p["_article_key"] = key
                p["title"] = meta["title"]
                p["source"] = meta["source"]
                p["published"] = meta["published"]
                p["final_url"] = key
                out_f.write(json.dumps(p, ensure_ascii=False) + "\n")
                written += 1
            if not places:
                out_f.write(json.dumps({"_article_key": key, "nama_tempat": None}, ensure_ascii=False) + "\n")
            already_done_manual.add(key)

print("Baris ditambahkan ke checkpoint:", written)
print("Lanjut ke sel pembacaan checkpoint di Bagian 4 (yang membangun df_places_raw).")

## 4. Jalankan Ekstraksi ke Semua Artikel (Opsi A: API / Opsi B: Regex Otomatis)

**Lewati bagian ini jika kamu memakai Opsi C (manual via chat Claude.ai) di Bagian 3c** — checkpoint sudah terisi dari sana, langsung lompat ke sel pembacaan checkpoint di bawah.

Sel di bawah menjalankan ekstraksi otomatis (LLM via API jika `ANTHROPIC_API_KEY` tersedia, atau fallback regex jika tidak). Setiap baris hasil = 1 tempat yang disebut di 1 artikel (`df_places_raw`). Progres disimpan berkala ke `places_raw_checkpoint.jsonl` supaya aman jika proses terhenti di tengah jalan (relevan terutama untuk pemanggilan LLM berbayar).

In [ ]:
CHECKPOINT_FILE = "places_raw_checkpoint.jsonl"
REQUEST_DELAY = 0.3

already_done = set()
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
                already_done.add(row["_article_key"])
            except Exception:
                continue
print("Artikel yang sudah diproses sebelumnya:", len(already_done))

with open(CHECKPOINT_FILE, "a", encoding="utf-8") as out_f:
    for _, row in tqdm(df_news_all.iterrows(), total=len(df_news_all)):
        key = row[dedup_key]
        if key in already_done:
            continue
        try:
            if USE_LLM:
                places = extract_places_llm(row.get("title", ""), row.get("article_text", ""))
            else:
                places = extract_places_fallback(row.get("title", ""), row.get("article_text", ""))
        except Exception as e:
            places = []
            print("Gagal ekstraksi:", key, "->", e)

        for p in places:
            p["_article_key"] = key
            p["title"] = row.get("title", "")
            p["source"] = row.get("source", "")
            p["published"] = row.get("published", "")
            p["final_url"] = row.get(dedup_key, "")
            out_f.write(json.dumps(p, ensure_ascii=False) + "\n")
        if not places:
            out_f.write(json.dumps({"_article_key": key, "nama_tempat": None}, ensure_ascii=False) + "\n")
        out_f.flush()
        if USE_LLM:
            time.sleep(REQUEST_DELAY)

print("Selesai. Baca ulang checkpoint untuk lanjut ke langkah berikutnya.")

In [ ]:
rows = []
with open(CHECKPOINT_FILE, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        if row.get("nama_tempat"):
            rows.append(row)

df_places_raw = pd.DataFrame(rows).drop_duplicates(subset=["_article_key", "nama_tempat"]).reset_index(drop=True)
print("Total baris tempat mentah (sebelum dedup antar-artikel):", len(df_places_raw))
df_places_raw.head(5)

## 5. Normalisasi & Deduplikasi Nama Tempat Antar-Artikel

Tempat yang sama sering disebut dengan variasi nama di artikel berbeda ("Amikan by Siki" vs "Amikan Cafe"). Dikelompokkan memakai fuzzy matching (`rapidfuzz`) + kecocokan kota/lokasi kasar, supaya sinyal "jumlah sumber unik" di langkah berikutnya akurat (satu tempat = satu grup, bukan tersebar jadi banyak grup kecil yang masing-masing terlihat 1 artikel).

In [ ]:
NAME_SIMILARITY_THRESHOLD = 85


def normalize_name(name: str) -> str:
    return re.sub(r"\s+", " ", str(name)).strip().lower()


df_places_raw["_norm_name"] = df_places_raw["nama_tempat"].map(normalize_name)

canonical_names = []  # list of representative names in order found
name_to_group = {}

for norm_name in df_places_raw["_norm_name"]:
    if norm_name in name_to_group:
        continue
    match_found = None
    for canon in canonical_names:
        if fuzz.token_sort_ratio(norm_name, canon) >= NAME_SIMILARITY_THRESHOLD:
            match_found = canon
            break
    if match_found:
        name_to_group[norm_name] = match_found
    else:
        canonical_names.append(norm_name)
        name_to_group[norm_name] = norm_name

df_places_raw["canonical_group"] = df_places_raw["_norm_name"].map(name_to_group)
print("Jumlah tempat unik setelah dedup:", df_places_raw["canonical_group"].nunique(),
      "(sebelumnya", df_places_raw["_norm_name"].nunique(), "nama mentah)")

## 6. Agregasi Sinyal per Tempat

Untuk setiap tempat unik, hitung:

- `jumlah_artikel`, `jumlah_source_unik` — proxy "banyak dibahas" tanpa API media sosial
- `rentang_hari_pemberitaan` — makin pendek rentang tanggal publikasi antar-artikel yang membahasnya, makin kuat indikasi *lonjakan* pemberitaan (bukan cuma tempat lama yang berulang-ulang direkomendasikan selama bertahun-tahun)
- `explicit_viral_ratio` — proporsi artikel yang eksplisit bilang tempat ini viral/ramai dibahas (bukan sekadar direkomendasikan)
- `earliest_article_date`, `indikasi_waktu_llm` (tanggal eksplisit tercatat dari LLM/regex, jika ada)

In [ ]:
def safe_parse_date(x):
    try:
        return dateparser.parse(str(x))
    except Exception:
        return None


df_places_raw["_pub_date"] = df_places_raw["published"].map(safe_parse_date)

agg_rows = []
for group, sub in df_places_raw.groupby("canonical_group"):
    dates = sub["_pub_date"].dropna()
    explicit_dates = sub["indikasi_waktu"].dropna()
    agg_rows.append({
        "canonical_group": group,
        "nama_tempat": sub["nama_tempat"].mode().iat[0] if not sub["nama_tempat"].mode().empty else sub["nama_tempat"].iat[0],
        "kategori": sub["kategori"].mode().iat[0] if not sub["kategori"].mode().empty else "",
        "lokasi_text": next((l for l in sub["lokasi_text"] if isinstance(l, str) and l.strip()), ""),
        "alasan_viral_contoh": next((a for a in sub["alasan_viral"] if isinstance(a, str) and a.strip()), ""),
        "jumlah_artikel": len(sub),
        "jumlah_source_unik": sub["source"].nunique(),
        "earliest_article_date": dates.min() if not dates.empty else None,
        "latest_article_date": dates.max() if not dates.empty else None,
        "rentang_hari_pemberitaan": (dates.max() - dates.min()).days if len(dates) > 1 else 0,
        "explicit_viral_ratio": sub["is_explicitly_viral"].fillna(False).mean(),
        "indikasi_waktu_eksplisit": explicit_dates.iat[0] if not explicit_dates.empty else None,
        "avg_confidence": sub["confidence"].astype(float).mean() if "confidence" in sub else None,
        "contoh_link": sub["final_url"].iat[0],
    })

df_places = pd.DataFrame(agg_rows).sort_values("jumlah_source_unik", ascending=False).reset_index(drop=True)
print("Total tempat unik:", len(df_places))
df_places.head(10)

## 7. Validasi Silang dengan Google Trends (Gratis)

**Kesulitan #1 & #2** — tanpa API media sosial berbayar, `pytrends` (wrapper tidak resmi Google Trends) dipakai sebagai proxy gratis untuk "seberapa ramai dibahas publik" dan "kapan mulai ramai":

- Ambil `interest_over_time` untuk nama tempat (12 bulan terakhir, region Indonesia).
- Cari minggu dengan lonjakan tertinggi (`trend_spike_score` = z-score puncak dibanding baseline) → `trend_spike_date`.
- Tempat yang benar-benar viral biasanya punya lonjakan tajam yang berdekatan waktunya dengan pemberitaan; tempat yang cuma "mainstream"/direkomendasikan berulang biasanya trennya flat.

Google Trends punya rate limit ketat & kadang gagal tanpa alasan jelas — dibungkus `try/except` + jeda antar-request, dan **hanya dijalankan untuk tempat dengan `jumlah_source_unik >= MIN_SOURCE_FOR_TRENDS`** supaya tidak membuang kuota untuk tempat yang jelas sekali disebut (kemungkinan besar noise).

In [ ]:
from pytrends.request import TrendReq

pytrends = TrendReq(hl="id-ID", tz=420)
MIN_SOURCE_FOR_TRENDS = 1  # naikkan (misal 2) untuk menghemat kuota di dataset besar
TRENDS_REQUEST_DELAY = 2.0


def get_trend_spike(query: str):
    try:
        pytrends.build_payload([query], timeframe="today 12-m", geo="ID")
        df_t = pytrends.interest_over_time()
        if df_t.empty:
            return None, None, None
        series = df_t[query]
        baseline = series.median()
        std = series.std() or 1
        peak_idx = series.idxmax()
        spike_score = (series.max() - baseline) / std
        return peak_idx.to_pydatetime(), float(spike_score), float(series.max())
    except Exception as e:
        return None, None, None


spike_dates, spike_scores, peak_values = [], [], []
for _, row in tqdm(df_places.iterrows(), total=len(df_places)):
    if row["jumlah_source_unik"] < MIN_SOURCE_FOR_TRENDS:
        spike_dates.append(None); spike_scores.append(None); peak_values.append(None)
        continue
    d, s, v = get_trend_spike(row["nama_tempat"])
    spike_dates.append(d); spike_scores.append(s); peak_values.append(v)
    time.sleep(TRENDS_REQUEST_DELAY)

df_places["trend_spike_date"] = spike_dates
df_places["trend_spike_score"] = spike_scores
df_places["trend_peak_value"] = peak_values
df_places.to_csv("df_places_with_trends_checkpoint.csv", index=False)
print("Selesai cek Google Trends. Checkpoint disimpan ke df_places_with_trends_checkpoint.csv")

## 8. Hitung `viral_confidence_score` & `viral_since_estimate`

Skor gabungan (0-1) dari 3 sinyal yang dinormalisasi:

- **source_signal** — `jumlah_source_unik` dinormalisasi (di-cap, lalu dibagi cap) → makin banyak media *berbeda* memberitakan dalam waktu berdekatan, makin kuat
- **explicit_signal** — `explicit_viral_ratio` (langsung 0-1)
- **trend_signal** — `trend_spike_score` dinormalisasi (di-cap ke skala wajar)

Bobot bisa disesuaikan di `WEIGHTS`. Tempat dengan skor rendah tapi `jumlah_artikel` tinggi kemungkinan besar cuma "mainstream"/rekomendasi berulang, bukan benar-benar viral → ditandai `mainstream_suspect = True` untuk direview manual.

`viral_since_estimate` diambil dengan prioritas: tanggal eksplisit dari artikel → tanggal spike Google Trends → tanggal artikel paling awal.

In [ ]:
WEIGHTS = {"source_signal": 0.4, "explicit_signal": 0.35, "trend_signal": 0.25}
SOURCE_CAP = 5       # >= 5 sumber unik dianggap signal maksimal
TREND_SCORE_CAP = 4.0  # z-score >= 4 dianggap lonjakan maksimal


def compute_row_score(row):
    source_signal = min(row["jumlah_source_unik"] / SOURCE_CAP, 1.0)
    explicit_signal = row["explicit_viral_ratio"] or 0.0
    trend_signal = min((row["trend_spike_score"] or 0) / TREND_SCORE_CAP, 1.0) if row["trend_spike_score"] else 0.0
    score = (
        WEIGHTS["source_signal"] * source_signal
        + WEIGHTS["explicit_signal"] * explicit_signal
        + WEIGHTS["trend_signal"] * trend_signal
    )
    return round(score, 3)


def compute_viral_since(row):
    if row.get("indikasi_waktu_eksplisit"):
        return row["indikasi_waktu_eksplisit"], "eksplisit_artikel"
    if row.get("trend_spike_date") is not None and pd.notna(row.get("trend_spike_date")):
        return row["trend_spike_date"], "google_trends_spike"
    if row.get("earliest_article_date") is not None and pd.notna(row.get("earliest_article_date")):
        return row["earliest_article_date"], "artikel_paling_awal"
    return None, "tidak_diketahui"


df_places["viral_confidence_score"] = df_places.apply(compute_row_score, axis=1)
since_results = df_places.apply(compute_viral_since, axis=1)
df_places["viral_since_estimate"] = since_results.map(lambda x: x[0])
df_places["viral_since_source"] = since_results.map(lambda x: x[1])

MAINSTREAM_SUSPECT_THRESHOLD = 0.35
df_places["mainstream_suspect"] = (
    (df_places["viral_confidence_score"] < MAINSTREAM_SUSPECT_THRESHOLD) & (df_places["jumlah_artikel"] >= 1)
)

df_places = df_places.sort_values("viral_confidence_score", ascending=False).reset_index(drop=True)
df_places[["nama_tempat", "jumlah_source_unik", "explicit_viral_ratio", "trend_spike_score",
           "viral_confidence_score", "viral_since_estimate", "viral_since_source", "mainstream_suspect"]].head(15)

## 9. Geocoding Tempat yang Lolos Threshold

Sama seperti notebook utama (Nominatim/OpenStreetMap dengan rate limiter), tapi hanya untuk tempat yang **tidak** ditandai `mainstream_suspect` (atau sesuaikan `MIN_SCORE_FOR_GEOCODE`) supaya tidak membuang kuota Nominatim untuk kandidat yang kemungkinan besar bukan benar-benar viral.

In [ ]:
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

MIN_SCORE_FOR_GEOCODE = 0.35

geolocator = Nominatim(user_agent="riset_tempat_viral_indonesia_v2")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1, max_retries=2, error_wait_seconds=2.0)


def geocode_place(nama_tempat: str, lokasi_text: str):
    query = f"{nama_tempat}, {lokasi_text}, Indonesia" if lokasi_text else f"{nama_tempat}, Indonesia"
    try:
        loc = geocode(query, language="id", addressdetails=True)
        if not loc:
            return {}
        addr = loc.raw.get("address", {})
        return {
            "latitude": loc.latitude,
            "longitude": loc.longitude,
            "kelurahan": addr.get("village") or addr.get("suburb"),
            "kecamatan": addr.get("city_district") or addr.get("suburb"),
            "kota_kabupaten": addr.get("city") or addr.get("county") or addr.get("regency"),
            "provinsi": addr.get("state"),
        }
    except Exception:
        return {}


to_geocode = df_places[df_places["viral_confidence_score"] >= MIN_SCORE_FOR_GEOCODE].copy()
geo_results = [geocode_place(r["nama_tempat"], r["lokasi_text"]) for _, r in tqdm(to_geocode.iterrows(), total=len(to_geocode))]
df_geo = pd.DataFrame(geo_results, index=to_geocode.index)
df_places_final = df_places.join(df_geo)
print("Berhasil geocoding:", df_places_final["latitude"].notna().sum(), "dari", len(to_geocode), "kandidat yang dicoba")

## 10. Simpan Hasil Akhir

In [ ]:
output_columns = [
    "nama_tempat", "kategori", "lokasi_text", "kelurahan", "kecamatan", "kota_kabupaten", "provinsi",
    "latitude", "longitude",
    "viral_since_estimate", "viral_since_source", "alasan_viral_contoh",
    "viral_confidence_score", "mainstream_suspect",
    "jumlah_artikel", "jumlah_source_unik", "explicit_viral_ratio", "trend_spike_score", "trend_spike_date",
    "earliest_article_date", "latest_article_date", "contoh_link",
]
output_columns = [c for c in output_columns if c in df_places_final.columns]

df_result_v2 = df_places_final[output_columns].sort_values("viral_confidence_score", ascending=False).reset_index(drop=True)
df_result_v2.to_csv("daftar_tempat_viral_v2.csv", index=False)
print("Tersimpan ke daftar_tempat_viral_v2.csv —", len(df_result_v2), "tempat")
df_result_v2.head(15)

## 11. Catatan, Keterbatasan & Langkah Lanjutan Opsional

**Tetap perlu review manual** untuk baris dengan `mainstream_suspect = True` atau `viral_since_source = "tidak_diketahui"`. Skor ini adalah *proxy*, bukan pengukuran langsung interaksi media sosial.

Keterbatasan yang masih ada:

- **Google Trends** mengukur volume *pencarian*, bukan volume *percakapan* di media sosial secara langsung — nama tempat yang generik ("Kopi Kenangan", "Taman Kota") bisa menghasilkan sinyal yang bias/noisy karena bercampur dengan pencarian tidak terkait. Pertimbangkan menambah kata kunci lokasi pada query trends (`f"{nama} {kota}"`) jika hasil terlihat mencurigakan.
- `pytrends` tidak resmi dan bisa terkena rate-limit (HTTP 429) — jika sering gagal, perbesar `TRENDS_REQUEST_DELAY` atau jalankan Bagian 7 di beberapa sesi terpisah.
- Ekstraksi LLM jauh lebih akurat dari regex tapi tetap bisa salah (halusinasi tempat, salah kategori) — `confidence` dari LLM bisa dipakai untuk filter tambahan (misal buang baris dengan `confidence < 0.5`).

Opsi tambahan yang **tidak** diimplementasikan di sini (butuh pertimbangan biaya/ToS lebih lanjut) untuk memperkuat sinyal viral tanpa API media sosial berbayar:

- **Reddit public search JSON** (`https://www.reddit.com/search.json?q=<nama_tempat>`) — gratis tanpa API key untuk pencarian dasar, bisa jadi sinyal tambahan volume perbincangan, meski komunitas Reddit Indonesia relatif kecil dibanding TikTok/Instagram/X untuk topik ini.
- **Sampling manual** pencarian hashtag TikTok/Instagram untuk beberapa kandidat teratas (bukan otomatis) sebagai validasi silang kualitatif, mengingat scraping otomatis platform tersebut melanggar ToS dan API resminya berbayar/terbatas.
- Jika anggaran API tersedia di masa depan, cross-check `viral_confidence_score` tertinggi dengan API resmi (misal X API tier berbayar) untuk kalibrasi bobot `WEIGHTS`.